# KV Training Dynamics Discovery

This notebook is artifact-first.

It does not assume a hardcoded model path or dataset path.
Instead it reads:

- `manifest.json`
- `train_history.jsonl`
- `summaries/checkpoint_index.csv`
- `battery/<checkpoint_id>/behavior.json`
- `battery/<checkpoint_id>/variable_scores.csv`
- `battery/<checkpoint_id>/variable_faithfulness.csv`
- `battery/<checkpoint_id>/operator_scores.csv`
- `battery/<checkpoint_id>/localization.csv`
- `battery/<checkpoint_id>/weight_metrics.json`
- `battery/<checkpoint_id>/neuron_scores.csv`
- `battery/<checkpoint_id>/feature_scores.csv` when `manifest["sae_tracking"]["enabled"]` is true
- `battery/<checkpoint_id>/superposition_metrics.json` when `manifest["sae_tracking"]["enabled"]` is true

If any required artifact is missing, the notebook fails immediately.


In [23]:
from __future__ import annotations

import json
from pathlib import Path

import pandas as pd

pd.set_option("display.max_colwidth", 200)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

RUN_DIR = Path("/Users/nelson/py/mechanistic_inter/runs/kv_textual_balanced_v1/pilot_seed0_curriculum_on_d64_l2")
CHECKPOINT_ID = "final_epoch_030"
MANIFEST_PATH = RUN_DIR / "manifest.json"
TRAIN_HISTORY_PATH = RUN_DIR / "train_history.jsonl"
CHECKPOINT_INDEX_PATH = RUN_DIR / "summaries" / "checkpoint_index.csv"
BATTERY_DIR = RUN_DIR / "battery" / CHECKPOINT_ID
manifest = json.loads(MANIFEST_PATH.read_text(encoding="utf-8"))
REQUIRED_PATHS = [
    MANIFEST_PATH,
    TRAIN_HISTORY_PATH,
    CHECKPOINT_INDEX_PATH,
    BATTERY_DIR / "behavior.json",
    BATTERY_DIR / "variable_scores.csv",
    BATTERY_DIR / "variable_faithfulness.csv",
    BATTERY_DIR / "operator_scores.csv",
    BATTERY_DIR / "localization.csv",
    BATTERY_DIR / "weight_metrics.json",
    BATTERY_DIR / "neuron_scores.csv",
]
if bool(manifest["sae_tracking"]["enabled"]):
    REQUIRED_PATHS.extend(
        [
            BATTERY_DIR / "feature_scores.csv",
            BATTERY_DIR / "superposition_metrics.json",
        ]
    )
for required_path in REQUIRED_PATHS:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required artifact: {required_path}")

train_history = pd.read_json(TRAIN_HISTORY_PATH, lines=True)
checkpoint_index = pd.read_csv(CHECKPOINT_INDEX_PATH)
behavior = json.loads((BATTERY_DIR / "behavior.json").read_text(encoding="utf-8"))
variable_scores = pd.read_csv(BATTERY_DIR / "variable_scores.csv")
variable_faithfulness = pd.read_csv(BATTERY_DIR / "variable_faithfulness.csv")
operator_scores = pd.read_csv(BATTERY_DIR / "operator_scores.csv")
localization_scores = pd.read_csv(BATTERY_DIR / "localization.csv")
weight_metrics = json.loads((BATTERY_DIR / "weight_metrics.json").read_text(encoding="utf-8"))
neuron_scores = pd.read_csv(BATTERY_DIR / "neuron_scores.csv")
feature_scores = None
superposition_metrics = None
if bool(manifest["sae_tracking"]["enabled"]):
    feature_scores = pd.read_csv(BATTERY_DIR / "feature_scores.csv")
    superposition_metrics = pd.read_json(BATTERY_DIR / "superposition_metrics.json")


## Run Manifest

The manifest is the source of truth for:

- dataset location
- model width/depth/head count
- curriculum mode
- checkpoint schedule
- battery limits
- birth thresholds


In [24]:
manifest


{'benchmark': {'name': 'kv_retrieval'},
 'dataset': {'dataset_dir': 'dataset/phase2/kv_retrieve_textual_balanced_v1',
  'train_split_by_pairs': {'2': 'train_2_pairs', '3': 'train_3_pairs'},
  'eval_splits': {'val': 'val', 'test': 'test', 'ood': 'test_ood_4_pairs'},
  'sweep_base_split': 'test'},
 'model': {'d_model': 64,
  'n_heads': 2,
  'd_ff': 128,
  'n_layers': 2,
  'max_seq_len': 16},
 'training': {'epochs': 30,
  'batch_size': 64,
  'learning_rate': 0.001,
  'weight_decay': 0.0,
  'seed': 0,
  'curriculum': 'on',
  'device': 'cpu'},
 'checkpoint_schedule': {'dense_through_epoch': 30,
  'log_spaced_epoch_count': 0,
  'save_epoch_zero': True,
  'save_final': True,
  'best_metric': 'val_accuracy'},
 'battery': {'train_probe_limit': 512,
  'sweep_base_limit': 128,
  'eval_batch_size': 256,
  'role_top_k': 3},
 'sae_tracking': {'enabled': True,
  'sites': ['block1_final_resid_after_mlp',
   'block1_final_mlp_out',
   'block1_head0_final_q',
   'block1_head0_final_v',
   'block1_head1_

## Checkpoint Index

This table tracks the saved checkpoints and the main birth-relevant metrics across time.


In [25]:
checkpoint_index


,run_dir,run_id,seed,intervention_signature,intervention_count,checkpoint_id,epoch,save_reason,val_accuracy,test_accuracy,ood_accuracy,query_key_site,query_key_score,query_key_family_min_score,query_key_faithfulness_site,query_key_faithfulness_score,query_key_faithfulness_family_min_score,matching_slot_site,matching_slot_score,matching_slot_family_min_score,matching_slot_faithfulness_site,matching_slot_faithfulness_score,matching_slot_faithfulness_family_min_score,selected_value_site,selected_value_score,selected_value_family_min_score,selected_value_faithfulness_site,selected_value_faithfulness_score,selected_value_faithfulness_family_min_score,routing_candidate,routing_score,routing_family_min_score,copy_candidate,copy_score,copy_family_min_score
0,/Users/nelson/py/mechanistic_inter/runs/kv_textual_balanced_v1/pilot_seed0_curriculum_on_d64_l2,pilot_seed0_curriculum_on_d64_l2,0,none,0,scheduled_epoch_000,0,scheduled,0.000000,0.000000,0.000000,block2_final_resid_after_mlp,0.996094,0.0,block2_final_resid_after_mlp,0.000000,0.000000,block2_final_mlp_out,0.337891,0.0,block2_final_mlp_out,0.000000,0.000000,block2_final_resid_after_mlp,0.356641,0.0,block2_final_resid_after_mlp,0.000000,0.0,block1_head1,0.360156,0.0,block1_head1,0.157031,0.0
1,/Users/nelson/py/mechanistic_inter/runs/kv_textual_balanced_v1/pilot_seed0_curriculum_on_d64_l2,pilot_seed0_curriculum_on_d64_l2,0,none,0,best_val_epoch_001,1,best_val,0.316333,0.320667,0.252667,block1_head1_final_head_out,0.833203,0.0,block1_head1_final_head_out,0.334635,0.333333,block2_head0_final_k,0.332813,0.0,block2_head0_final_k,0.332031,0.166667,block1_head0_final_head_out,0.348047,0.0,block1_head0_final_head_out,0.339844,0.0,block1_head0,0.346094,0.0,block2_head1,0.835547,0.0
2,/Users/nelson/py/mechanistic_inter/runs/kv_textual_balanced_v1/pilot_seed0_curriculum_on_d64_l2,pilot_seed0_curriculum_on_d64_l2,0,none,0,scheduled_epoch_001,1,scheduled,0.316333,0.320667,0.252667,block1_head1_final_head_out,0.833203,0.0,block1_head1_final_head_out,0.334635,0.333333,block2_head0_final_k,0.332813,0.0,block2_head0_final_k,0.332031,0.166667,block1_head0_final_head_out,0.348047,0.0,block1_head0_final_head_out,0.339844,0.0,block1_head0,0.346094,0.0,block2_head1,0.835547,0.0
3,/Users/nelson/py/mechanistic_inter/runs/kv_textual_balanced_v1/pilot_seed0_curriculum_on_d64_l2,pilot_seed0_curriculum_on_d64_l2,0,none,0,best_val_epoch_002,2,best_val,0.327667,0.326333,0.245667,block1_final_resid_after_mlp,0.741016,0.0,block1_final_resid_after_mlp,0.333333,0.333333,block1_final_resid_after_mlp,0.343359,0.0,block1_final_resid_after_mlp,0.333333,0.333333,block1_head0_final_head_out,0.350000,0.0,block1_head0_final_head_out,0.371094,0.0,block1_head0,0.340625,0.0,block2_head1,0.842969,0.0
4,/Users/nelson/py/mechanistic_inter/runs/kv_textual_balanced_v1/pilot_seed0_curriculum_on_d64_l2,pilot_seed0_curriculum_on_d64_l2,0,none,0,scheduled_epoch_002,2,scheduled,0.327667,0.326333,0.245667,block1_final_resid_after_mlp,0.741016,0.0,block1_final_resid_after_mlp,0.333333,0.333333,block1_final_resid_after_mlp,0.343359,0.0,block1_final_resid_after_mlp,0.333333,0.333333,block1_head0_final_head_out,0.350000,0.0,block1_head0_final_head_out,0.371094,0.0,block1_head0,0.340625,0.0,block2_head1,0.842969,0.0
5,/Users/nelson/py/mechanistic_inter/runs/kv_textual_balanced_v1/pilot_seed0_curriculum_on_d64_l2,pilot_seed0_curriculum_on_d64_l2,0,none,0,scheduled_epoch_003,3,scheduled,0.327000,0.334667,0.243000,block1_final_resid_after_mlp,0.712500,0.0,block1_final_resid_after_mlp,0.332031,0.166667,block1_final_resid_after_mlp,0.347266,0.0,block1_final_resid_after_mlp,0.332031,0.166667,block1_head0_final_head_out,0.362109,0.0,block1_head0_final_head_out,0.367188,0.0,block1_head0,0.339844,0.0,block2_head1,0.817578,0.0
6,/Users/nelson/py/mechanistic_inter/runs/kv_textual_balanced_v1/pilot_seed0_curriculum_on_d64_l2,pilot_seed0_curriculum_on_d64_l2,0,none,0,best_val_epoch_004,4,best_val,0.328000,0.319333,0.244667,block1_head0_final_head_out,0.69

## Per-Step Dynamics

This is the cheap-every-step telemetry:

- batch loss
- total grad norm
- total update norm
- per-matrix and per-head-slice parameter metrics


In [26]:
train_history[[
    "epoch",
    "global_step",
    "batch_index",
    "batch_loss",
    "curriculum_stage",
    "total_grad_norm",
    "total_update_norm",
    "relative_total_update_norm",
]]


,epoch,global_step,batch_index,batch_loss,curriculum_stage,total_grad_norm,total_update_norm,relative_total_update_norm
0,1,0,0,53.649643,2_pairs_only,26.094112,0.342507,0.006039
1,1,1,1,49.212914,2_pairs_only,32.481378,0.298778,0.005268
2,1,2,2,45.377594,2_pairs_only,42.618760,0.285589,0.005035
3,1,3,3,40.334835,2_pairs_only,49.551565,0.283974,0.005005
4,1,4,4,33.003220,2_pairs_only,54.366428,0.281076,0.004953
...,...,...,...,...,...,...,...,...
14065,30,14065,464,0.003057,2_pairs_only,0.215760,0.041669,0.000681
14066,30,14066,465,0.055149,2_pairs_only,1.065420,0.043025,0.000703
14067,30,14067,466,0.002232,2_pairs_only,0.071517,0.038880,0.000635
14068,30,14068,467,0.037274,2_pairs_only,0.891408,0.040611,0.000664


In [27]:
train_history.iloc[0]["parameter_metrics"][:12]


[{'name': 'token_embed.weight',
  'layer_index': None,
  'head_index': None,
  'component': 'token_embed',
  'shape': [36, 64],
  'param_norm_pre': 48.42069625854492,
  'param_norm_post': 48.419803619384766,
  'grad_norm': 8.972235679626465,
  'update_norm': 0.047999732196331,
  'relative_update_norm': 0.0009913061129900001,
  'cosine_to_previous': 0.9999993341991231},
 {'name': 'block1.norm1.weight',
  'layer_index': 0,
  'head_index': None,
  'component': 'norm1',
  'shape': [64],
  'param_norm_pre': 8.0,
  'param_norm_post': 8.002503395080566,
  'grad_norm': 0.5428395271301271,
  'update_norm': 0.008000208996236,
  'relative_update_norm': 0.0010000261245290002,
  'cosine_to_previous': 0.9999996424840061},
 {'name': 'block1.attn.q_proj.weight',
  'layer_index': 0,
  'head_index': None,
  'component': 'q_proj',
  'shape': [64, 64],
  'param_norm_pre': 4.5802578926086435,
  'param_norm_post': 4.581172466278076,
  'grad_norm': 2.008671522140503,
  'update_norm': 0.063999712467193,
  're

## Behavior

Split-level metrics plus controlled prompt-family breakdown.


In [28]:
pd.DataFrame(
    [
        {"split": split_name, **metrics}
        for split_name, metrics in behavior["split_metrics"].items()
    ]
)


,split,rows,loss,accuracy,margin
0,test,3000,0.333294,0.934667,8.521207
1,test_ood_4_pairs,3000,0.996484,0.845667,6.539952
2,train_2_pairs,30000,0.014011,0.995100,10.674455
3,train_3_pairs,30000,0.319682,0.937567,8.693167
4,val,3000,0.345855,0.939333,8.737222


In [29]:
pd.DataFrame(behavior["family_breakdown"]).sort_values(["accuracy", "margin"], ascending=[False, False])


,family_name,rows,accuracy,margin
3,same_slot_different_answer,256,0.941406,8.825551
1,query_key_sweep,384,0.914062,8.319994
2,same_answer_different_slot,256,0.910156,8.595732
4,slot_permutation,768,0.902344,8.465047
5,value_permutation,768,0.902344,8.090634
0,longer_context_ood,128,0.843750,6.506338


## Variable Recovery

The summary rows report the best site for each variable, along with pooled and family-min scores.


In [30]:
variable_scores.query("row_kind == 'summary'")


,site,variable,train_rows,eval_rows,num_classes,chance_accuracy,eval_accuracy,eval_margin_over_chance,weight_norm,row_kind,best_site,pooled_score,family_min_score,family_mean_score,family_max_score
75,NaN,query_key,NaN,NaN,NaN,NaN,NaN,NaN,NaN,summary,block1_head0_final_head_out,0.994531,0.0,0.982319,1.0
76,NaN,matching_slot,NaN,NaN,NaN,NaN,NaN,NaN,NaN,summary,block2_head1_final_head_out,0.450391,0.0,0.373972,1.0
77,NaN,selected_value,NaN,NaN,NaN,NaN,NaN,NaN,NaN,summary,block2_head1_final_head_out,0.915625,0.0,0.903783,1.0


In [31]:
variable_scores.query("row_kind == 'probe'").sort_values(
    ["variable", "eval_accuracy", "eval_margin_over_chance"],
    ascending=[True, False, False],
).head(24)


,site,variable,train_rows,eval_rows,num_classes,chance_accuracy,eval_accuracy,eval_margin_over_chance,weight_norm,row_kind,best_site,pooled_score,family_min_score,family_mean_score,family_max_score
0,block2_head1_final_head_out,matching_slot,512.0,2560.0,3.0,0.333333,0.450391,0.117057,4.555066,probe,NaN,NaN,NaN,NaN,NaN
1,block2_head1_final_resid_contribution,matching_slot,512.0,2560.0,3.0,0.333333,0.450391,0.117057,15.640809,probe,NaN,NaN,NaN,NaN,NaN
2,final_hidden,matching_slot,512.0,2560.0,3.0,0.333333,0.427734,0.094401,6.324738,probe,NaN,NaN,NaN,NaN,NaN
3,block2_final_resid_after_mlp,matching_slot,512.0,2560.0,3.0,0.333333,0.426172,0.092839,7.839356,probe,NaN,NaN,NaN,NaN,NaN
4,block2_final_mlp_out,matching_slot,512.0,2560.0,3.0,0.333333,0.376172,0.042839,12.499414,probe,NaN,NaN,NaN,NaN,NaN
5,block2_head0_final_head_out,matching_slot,512.0,2560.0,3.0,0.333333,0.374609,0.041276,1.953911,probe,NaN,NaN,NaN,NaN,NaN
6,block2_head0_final_resid_contribution,matching_slot,512.0,2560.0,3.0,0.333333,0.374219,0.040885,5.149446,probe,NaN,NaN,NaN,NaN,NaN
7,block2_head0_final_v,matching_slot,512.0,2560.0,3.0,0.333333,0.335156,0.001823,5.955019,probe,NaN,NaN,NaN,NaN,NaN
8,block2_head1_final_q,matching_slot,512.0,2560.0,3.0,0.333333,0.331641,-0.001693,25.913784,probe,NaN,NaN,NaN,NaN,NaN
9,block1_head0_final_q,matching_slot,512.0,2560.0,3.0,0.333333,0.330078,-0.003255,0.766514,probe,NaN,NaN,NaN,NaN,NaN


## Variable Faithfulness

These rows test whether patching the best site for a variable from one sweep example into another
causes the answer to follow the symbolic intervention.


In [32]:
variable_faithfulness.query("row_kind == 'summary'")


,row_kind,variable,site,family_name,base_prompt_id,base_prompt,source_prompt,base_variable_value,source_variable_value,expected_target,predicted_token,target_token,foil_token,margin,correct,rows,family_accuracy,family_margin_mean,pooled_score,pooled_margin_mean,family_min_score,family_mean_score,family_max_score
2176,summary,query_key,block1_head0_final_head_out,query_key_sweep,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,768.0,NaN,NaN,0.912760,8.170609,0.333333,0.912760,1.0
2177,summary,matching_slot,block2_head1_final_head_out,query_key_sweep,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,768.0,NaN,NaN,0.789062,4.405041,0.333333,0.789062,1.0
2178,summary,selected_value,block2_head1_final_head_out,same_slot_different_answer,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,256.0,NaN,NaN,0.792969,4.642851,0.000000,0.792969,1.0


In [33]:
variable_faithfulness.query("row_kind == 'family_summary'")


,row_kind,variable,site,family_name,base_prompt_id,base_prompt,source_prompt,base_variable_value,source_variable_value,expected_target,predicted_token,target_token,foil_token,margin,correct,rows,family_accuracy,family_margin_mean,pooled_score,pooled_margin_mean,family_min_score,family_mean_score,family_max_score
1792,family_summary,query_key,block1_head0_final_head_out,query_key_sweep,test_000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,1.0,6.613220,NaN,NaN,NaN,NaN,NaN
1793,family_summary,query_key,block1_head0_final_head_out,query_key_sweep,test_000001,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,1.0,7.988113,NaN,NaN,NaN,NaN,NaN
1794,family_summary,query_key,block1_head0_final_head_out,query_key_sweep,test_000002,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,1.0,7.699046,NaN,NaN,NaN,NaN,NaN
1795,family_summary,query_key,block1_head0_final_head_out,query_key_sweep,test_000003,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,1.0,11.756514,NaN,NaN,NaN,NaN,NaN
1796,family_summary,query_key,block1_head0_final_head_out,query_key_sweep,test_000004,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,6.0,1.0,11.403263,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2171,family_summary,selected_value,block2_head1_final_head_out,same_slot_different_answer,test_000123,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,0.5,3.719199,NaN,NaN,NaN,NaN,NaN
2172,family_summary,selected_value,block2_head1_final_head_out,same_slot_different_answer,test_000124,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,10.952870,NaN,NaN,NaN,NaN,NaN
2173,family_summary,selected_value,block2_head1_final_head_out,same_slot_different_answer,test_000125,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,11.419756,NaN,NaN,NaN,NaN,NaN
2174,family_summary,selected_value,block2_head1_final_head_out,same_slot_different_answer,test_000126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2.0,1.0,5.103088,NaN,NaN,NaN,NaN,NaN


## Operator Scores

Every head is scored for routing-like and copy-like behavior.
The candidate rows identify the selected routing and copy heads at this checkpoint.


In [34]:
operator_scores.query("row_kind == 'candidate'")


,row_kind,layer_index,head_index,head_name,routing_score,routing_key_score,routing_value_score,routing_family_min_score,copy_score,copy_family_min_score,copy_rank_mean,candidate_type,supported,reason,routing_candidate,copy_candidate
4,candidate,1.0,1.0,block2_head1,0.905469,0.617578,0.892578,0.0,0.991016,0.0,1.010156,routing,NaN,NaN,NaN,NaN
5,candidate,1.0,1.0,block2_head1,0.905469,0.617578,0.892578,0.0,0.991016,0.0,1.010156,copy,NaN,NaN,NaN,NaN


In [35]:
operator_scores.query("row_kind == 'head_score'").sort_values(
    ["routing_score", "copy_score"],
    ascending=[False, False],
).head(24)


,row_kind,layer_index,head_index,head_name,routing_score,routing_key_score,routing_value_score,routing_family_min_score,copy_score,copy_family_min_score,copy_rank_mean,candidate_type,supported,reason,routing_candidate,copy_candidate
3,head_score,1.0,1.0,block2_head1,0.905469,0.617578,0.892578,0.0,0.991016,0.0,1.010156,NaN,NaN,NaN,NaN,NaN
2,head_score,1.0,0.0,block2_head0,0.786719,0.580078,0.786328,0.0,0.428125,0.0,4.993750,NaN,NaN,NaN,NaN,NaN
0,head_score,0.0,0.0,block1_head0,0.330078,0.332031,0.335156,0.0,0.717187,0.0,1.932813,NaN,NaN,NaN,NaN,NaN
1,head_score,0.0,1.0,block1_head1,0.305078,0.325391,0.303906,0.0,0.534375,0.0,3.371875,NaN,NaN,NaN,NaN,NaN


In [36]:
operator_scores.query("row_kind != 'head_score'")


,row_kind,layer_index,head_index,head_name,routing_score,routing_key_score,routing_value_score,routing_family_min_score,copy_score,copy_family_min_score,copy_rank_mean,candidate_type,supported,reason,routing_candidate,copy_candidate
4,candidate,1.0,1.0,block2_head1,0.905469,0.617578,0.892578,0.0,0.991016,0.0,1.010156,routing,NaN,NaN,NaN,NaN
5,candidate,1.0,1.0,block2_head1,0.905469,0.617578,0.892578,0.0,0.991016,0.0,1.010156,copy,NaN,NaN,NaN,NaN
6,status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,Selected routing and copy candidates are not adjacent layers,block2_head1,block2_head1


## Localization

This combines per-head final-position ablations with any path-patching rows available for the selected candidate pair.


In [37]:
localization_scores


,row_kind,layer_index,head_index,head_name,baseline_accuracy,ablated_accuracy,accuracy_drop,baseline_margin,ablated_margin,margin_drop,supported,reason,routing_candidate,copy_candidate
0,head_ablation,0.0,0.0,block1_head0,0.905859,0.239063,0.666797,8.282149,-4.290324,12.572472,NaN,NaN,NaN,NaN
1,head_ablation,0.0,1.0,block1_head1,0.905859,0.898828,0.007031,8.282149,7.794821,0.487328,NaN,NaN,NaN,NaN
2,head_ablation,1.0,0.0,block2_head0,0.905859,0.858984,0.046875,8.282149,6.066484,2.215665,NaN,NaN,NaN,NaN
3,head_ablation,1.0,1.0,block2_head1,0.905859,0.371094,0.534766,8.282149,-1.834418,10.116566,NaN,NaN,NaN,NaN
4,status,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,False,Selected routing and copy candidates are not adjacent layers,block2_head1,block2_head1


## Weight Metrics

These metrics summarize whole-matrix and per-head slice geometry at the selected checkpoint.


In [38]:
pd.DataFrame(weight_metrics["matrices"]).head(20)


,name,layer_index,head_index,shape,fro_norm,spectral_norm,top_singular_values
0,q_proj,0,None,"[64, 64]",6.347409,3.060375,"[3.0603749752044678, 1.7923216819763184, 1.6437973976135254, 1.3806103467941284, 1.276274561882019]"
1,k_proj,0,None,"[64, 64]",6.531615,2.793033,"[2.793032646179199, 2.3081722259521484, 1.6791791915893555, 1.6249696016311646, 1.337127685546875]"
2,v_proj,0,None,"[64, 64]",7.075062,2.390628,"[2.3906283378601074, 2.195967674255371, 2.025482654571533, 1.846793532371521, 1.694199562072754]"
3,o_proj,0,None,"[64, 64]",6.991288,2.640257,"[2.6402571201324463, 2.2906267642974854, 1.9326317310333252, 1.8607805967330933, 1.6289550065994263]"
4,gate_proj,0,None,"[128, 64]",8.672818,3.030898,"[3.030897855758667, 2.2384705543518066, 2.1098473072052, 1.806628942489624, 1.7578436136245728]"
5,up_proj,0,None,"[128, 64]",8.284198,2.310825,"[2.3108253479003906, 2.116029739379883, 1.9484177827835083, 1.867366075515747, 1.7194709777832031]"
6,down_proj,0,None,"[64, 128]",7.799891,4.071222,"[4.071221828460693, 2.3353209495544434, 1.8905421495437622, 1.7927502393722534, 1.6796917915344238]"
7,q_proj,1,None,"[64, 64]",8.332216,3.608147,"[3.6081466674804688, 3.198704719543457, 2.919543743133545, 2.312685251235962, 2.098609447479248]"
8,k_proj,1,None,"[64, 64]",6.808949,2.723839,"[2.723839044570923, 2.4457831382751465, 2.1689298152923584, 2.0096869468688965, 1.794187307357788]"
9,v_proj,1,None,"[64, 64]",8.031729,2.312086,"[2.3120856285095215, 2.2439324855804443, 2.1783201694488525, 2.1151323318481445, 2.038714647293091]"


In [39]:
pd.DataFrame(weight_metrics["head_slices"]).head(20)


,name,layer_index,head_index,shape,fro_norm,spectral_norm,top_singular_values
0,q_proj_head_slice,0,0,"[32, 64]",4.603010,2.560383,"[2.5603830814361572, 1.3459323644638062, 1.2016100883483887, 1.1861991882324219, 1.0192371606826782]"
1,k_proj_head_slice,0,0,"[32, 64]",4.598547,2.569827,"[2.569827079772949, 1.4720783233642578, 1.2541779279708862, 1.0569711923599243, 0.9765466451644897]"
2,v_proj_head_slice,0,0,"[32, 64]",5.544128,2.273340,"[2.2733404636383057, 1.9252285957336426, 1.8797366619110107, 1.6511576175689697, 1.5619571208953857]"
3,o_proj_head_slice,0,0,"[64, 32]",5.397244,2.211849,"[2.2118489742279053, 2.003222703933716, 1.811279296875, 1.6457747220993042, 1.5130380392074585]"
4,q_proj_head_slice,0,1,"[32, 64]",4.370572,1.954963,"[1.9549634456634521, 1.4449026584625244, 1.2537033557891846, 1.094291090965271, 0.9880173802375793]"
5,k_proj_head_slice,0,1,"[32, 64]",4.638466,2.270291,"[2.2702910900115967, 1.6482328176498413, 1.3011083602905273, 1.1555635929107666, 1.06709885597229]"
6,v_proj_head_slice,0,1,"[32, 64]",4.395356,1.536434,"[1.5364338159561157, 1.3582935333251953, 1.2305560111999512, 1.1564441919326782, 1.1274946928024292]"
7,o_proj_head_slice,0,1,"[64, 32]",4.443858,2.039677,"[2.0396766662597656, 1.381208062171936, 1.284009575843811, 1.1485649347305298, 1.0582414865493774]"
8,q_proj_head_slice,1,0,"[32, 64]",5.697852,2.785941,"[2.7859408855438232, 2.220612049102783, 2.0126852989196777, 1.6228864192962646, 1.3806082010269165]"
9,k_proj_head_slice,1,0,"[32, 64]",4.861663,2.172426,"[2.1724255084991455, 1.952850580215454, 1.7270517349243164, 1.3628441095352173, 1.0697104930877686]"


## Neuron Tracking

These rows show which MLP neurons are becoming selective for query key, matching slot, or selected value.


In [40]:
neuron_scores.query("row_kind == 'layer_top'")


,layer_index,layer_name,neuron_index,component,mean_activation,mean_abs_activation,activation_std,positive_rate,nonzero_rate,max_activation,min_activation,write_norm,activation_write_product,query_key_eta2,query_key_group_gap,matching_slot_eta2,matching_slot_group_gap,selected_value_eta2,selected_value_group_gap,best_variable,best_selectivity_score,best_variable_group_gap,row_kind
256,0,block1,106,block1.mlp.neuron_106,-0.348712,0.626195,1.412523,0.494141,1.0,0.703413,-5.296329,0.803834,0.503357,0.994107,5.494757,0.001029,0.331342,0.036471,1.173872,query_key,0.994107,5.494757,layer_top
257,0,block1,7,block1.mlp.neuron_7,0.330605,0.500615,1.250129,0.360938,1.0,5.640233,-0.430895,0.690865,0.345858,0.987349,5.671167,0.000369,0.143446,0.030358,0.870238,query_key,0.987349,5.671167,layer_top
258,0,block1,99,block1.mlp.neuron_99,0.071933,0.127776,0.163618,0.620703,1.0,0.529261,-0.181423,0.549639,0.070231,0.986016,0.603006,0.002155,0.034211,0.036026,0.086982,query_key,0.986016,0.603006,layer_top
259,0,block1,26,block1.mlp.neuron_26,-0.066910,0.726945,1.189848,0.769141,1.0,1.059407,-4.180999,0.806644,0.586385,0.975835,4.577487,0.001426,0.304561,0.030903,0.824962,query_key,0.975835,4.577487,layer_top
260,0,block1,83,block1.mlp.neuron_83,0.043708,0.132823,0.189828,0.634375,1.0,0.498975,-0.549209,0.574618,0.076322,0.974844,0.906960,0.001166,0.023109,0.050975,0.171418,query_key,0.974844,0.906960,layer_top
261,0,block1,68,block1.mlp.neuron_68,-0.003660,0.184628,0.251611,0.360156,1.0,0.719269,-0.400647,0.570869,0.105398,0.969942,0.923248,0.001580,0.037460,0.032030,0.172276,query_key,0.969942,0.923248,layer_top
262,0,block1,75,block1.mlp.neuron_75,0.118408,0.416677,0.555033,0.446094,1.0,1.573827,-0.819319,0.887311,0.369722,0.965967,2.069026,0.003753,0.185931,0.030364,0.474356,query_key,0.965967,2.069026,layer_top
263,0,block1,32,block1.mlp.neuron_32,0.048890,0.161143,0.182202,0.639063,1.0,0.418134,-0.327728,0.564315,0.090935,0.963204,0.612277,0.001344,0.017059,0.060781,0.164869,query_key,0.963204,0.612277,layer_top
264,0,block1,86,block1.mlp.neuron_86,0.051691,0.130697,0.159269,0.695312,1.0,0.350920,-0.492589,0.615194,0.080404,0.960010,0.648914,0.001087,0.021404,0.058069,0.129695,query_key,0.960010,0.648914,layer_top
265,0,block1,60,block1.mlp.neuron_60,-5.320977,5.320977,4.885234,0.000000,1.0,-0.180729,-17.298874,0.989030,5.262605,0.959558,13.424114,0.001227,0.923607,0.047456,4.101556,query_key,0.959558,13.424114,layer_top


In [41]:
neuron_scores.query("row_kind == 'neuron'").sort_values(
    ["layer_index", "best_selectivity_score", "activation_write_product"],
    ascending=[True, False, False],
).head(30)


,layer_index,layer_name,neuron_index,component,mean_activation,mean_abs_activation,activation_std,positive_rate,nonzero_rate,max_activation,min_activation,write_norm,activation_write_product,query_key_eta2,query_key_group_gap,matching_slot_eta2,matching_slot_group_gap,selected_value_eta2,selected_value_group_gap,best_variable,best_selectivity_score,best_variable_group_gap,row_kind
0,0,block1,106,block1.mlp.neuron_106,-0.348712,0.626195,1.412523,0.494141,1.0,0.703413,-5.296329,0.803834,0.503357,0.994107,5.494757,0.001029,0.331342,0.036471,1.173872,query_key,0.994107,5.494757,neuron
1,0,block1,7,block1.mlp.neuron_7,0.330605,0.500615,1.250129,0.360938,1.0,5.640233,-0.430895,0.690865,0.345858,0.987349,5.671167,0.000369,0.143446,0.030358,0.870238,query_key,0.987349,5.671167,neuron
2,0,block1,99,block1.mlp.neuron_99,0.071933,0.127776,0.163618,0.620703,1.0,0.529261,-0.181423,0.549639,0.070231,0.986016,0.603006,0.002155,0.034211,0.036026,0.086982,query_key,0.986016,0.603006,neuron
3,0,block1,26,block1.mlp.neuron_26,-0.066910,0.726945,1.189848,0.769141,1.0,1.059407,-4.180999,0.806644,0.586385,0.975835,4.577487,0.001426,0.304561,0.030903,0.824962,query_key,0.975835,4.577487,neuron
4,0,block1,83,block1.mlp.neuron_83,0.043708,0.132823,0.189828,0.634375,1.0,0.498975,-0.549209,0.574618,0.076322,0.974844,0.906960,0.001166,0.023109,0.050975,0.171418,query_key,0.974844,0.906960,neuron
5,0,block1,68,block1.mlp.neuron_68,-0.003660,0.184628,0.251611,0.360156,1.0,0.719269,-0.400647,0.570869,0.105398,0.969942,0.923248,0.001580,0.037460,0.032030,0.172276,query_key,0.969942,0.923248,neuron
6,0,block1,75,block1.mlp.neuron_75,0.118408,0.416677,0.555033,0.446094,1.0,1.573827,-0.819319,0.887311,0.369722,0.965967,2.069026,0.003753,0.185931,0.030364,0.474356,query_key,0.965967,2.069026,neuron
7,0,block1,32,block1.mlp.neuron_32,0.048890,0.161143,0.182202,0.639063,1.0,0.418134,-0.327728,0.564315,0.090935,0.963204,0.612277,0.001344,0.017059,0.060781,0.164869,query_key,0.963204,0.612277,neuron
8,0,block1,86,block1.mlp.neuron_86,0.051691,0.130697,0.159269,0.695312,1.0,0.350920,-0.492589,0.615194,0.080404,0.960010,0.648914,0.001087,0.021404,0.058069,0.129695,query_key,0.960010,0.648914,neuron
9,0,block1,60,block1.mlp.neuron_60,-5.320977,5.320977,4.885234,0.000000,1.0,-0.180729,-17.298874,0.989030,5.262605,0.959558,13.424114,0.001227,0.923607,0.047456,4.101556,query_key,0.959558,13.424114,neuron


## SAE Feature Tracking

When SAE tracking is enabled in the manifest, this section shows checkpoint-local sparse feature summaries
over the configured dynamic sites.


In [42]:
if feature_scores is None:
    "SAE tracking disabled in manifest"
else:
    feature_scores.query("row_kind == 'site_summary'")


In [43]:
if feature_scores is None:
    "SAE tracking disabled in manifest"
else:
    feature_scores.query("row_kind == 'feature'").sort_values(
        ["site", "best_selectivity_score", "mean_abs_activation"],
        ascending=[True, False, False],
    ).head(30)


## Superposition Metrics

These metrics summarize feature density, decoder overlap, and decoder rank geometry for each tracked site.


In [44]:
if superposition_metrics is None:
    "SAE tracking disabled in manifest"
else:
    superposition_metrics
